In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Hybrid search — BM25 + dense, fused with RRF

Dense retrieval matches *meaning* but can fumble exact tokens — error codes,
policy IDs, product names. Lexical **BM25** nails those but misses paraphrases.
Real systems run both and fuse the results. We build BM25 from scratch, then
fuse with **Reciprocal Rank Fusion**.


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
from ragkit.reference import structure_aware_chunks   # from nb 02
from ragkit.corpus import Chunk

docs = load_documents()
chunks = []
for doc_id, text in docs.items():
    for i, (heading, ctext) in enumerate(
        [(c.section, c.text) for c in structure_aware_chunks(doc_id, text)]
    ):
        chunks.append(Chunk(f"{doc_id}#{i}", doc_id, ctext, heading))
print(len(chunks), "chunks indexed")

### Exercise 1 — BM25 scoring

BM25 scores a query term `t` in document `i` as

```
idf(t) · f(t,i)·(k1+1) / ( f(t,i) + k1·(1 − b + b·|d_i|/avgdl) )
```

where `f(t,i)` is the term's frequency in doc `i`, `|d_i|` its length, `avgdl`
the mean length. Fill in that formula (the `idf` and counts are precomputed).


In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = corpus_tokens
        self.N = len(corpus_tokens)
        self.avgdl = sum(len(d) for d in corpus_tokens) / self.N
        self.tf = [Counter(d) for d in corpus_tokens]
        df = Counter()
        for d in corpus_tokens:
            df.update(set(d))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}

    def score(self, q_tokens, i):
        dl = len(self.docs[i])
        s = 0.0
        for t in q_tokens:
            f = self.tf[i].get(t, 0)
            if f == 0:
                continue
            numerator = f * (self.k1 + 1)
            denominator = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += self.idf.get(t, 0.0) * numerator / denominator
        return s

    def search(self, query, k=5):
        q = tokenize(query)
        ranked = sorted(((i, self.score(q, i)) for i in range(self.N)),
                        key=lambda x: x[1], reverse=True)
        return ranked[:k]

bm = BM25([tokenize(c.text) for c in chunks])
# hand-checkable: more occurrences of a term -> higher score, absent term -> 0
toy = BM25([["cat", "cat", "mat"], ["cat", "hat"], ["dog"]])
assert toy.score(["cat"], 0) > toy.score(["cat"], 1) > 0
assert toy.score(["cat"], 2) == 0.0
# on the real corpus, an exact code lands the right doc
top_i, _ = bm.search("what does ERR_4290 mean", k=1)[0]
print("ERR_4290 ->", chunks[top_i].doc_id)

### See where each method wins

Build a dense index over the same chunks and compare the two retrievers on a
**lexical** query (exact code) and a **semantic** query (paraphrase).


In [ ]:
emb = get_embedder()
mat = emb.encode([c.text for c in chunks])

def dense_ids(query, k=5):
    qv = emb.encode(query)
    order = np.argsort(-(mat @ qv))[:k]
    return [chunks[i].chunk_id for i in order]

def bm25_ids(query, k=5):
    return [chunks[i].chunk_id for i, _ in bm.search(query, k)]

for q in ["what does ERR_4290 mean",                       # lexical
          "am I allowed to work from home every day"]:      # semantic
    print(f"\nQ: {q}")
    print("  dense:", [i.split('#')[0] for i in dense_ids(q, 3)])
    print("  bm25 :", [i.split('#')[0] for i in bm25_ids(q, 3)])

### Exercise 2 — Reciprocal Rank Fusion

RRF combines ranked lists using only **rank position**, so it doesn't care that
BM25 and cosine scores live on different scales:

```
score(d) = Σ_lists 1 / (k + r(d))      # k = 60; r(d) = 1 for the top result of a list
```

(`enumerate` counts from 0, so in code that is `1 / (k + rank + 1)`.)

Implement it, then fuse the dense and BM25 rankings.


In [ ]:
from collections import defaultdict

def rrf(rankings, k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += 1.0 / (k + rank + 1)
    return [d for d, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

# consensus check: 'b' is near the top of both lists, so it should win
fused = rrf([["a", "b", "c"], ["b", "c", "a"]])
assert fused == ["b", "a", "c"], fused   # 1/61+1/62 > 1/61+1/63 > 1/62+1/63

# exact-formula check. With k=60 an off-by-one in the rank barely moves a score, so
# the probe needs deep ranks. Four documents, filler everywhere else:
#   x1: 1st in A                  -> 1/61          = 0.016393
#   y1: 38th in B and 100th in A  -> 1/98 + 1/160  = 0.016454
#   x2: 2nd in B                  -> 1/62          = 0.016129
#   y2: 40th in A and 98th in B   -> 1/100 + 1/158 = 0.016329
# Ranks counted from 0 put x1 above y1; ranks from 2 put y2 above x1.
def ranked(placed, n, filler):
    return [placed.get(r, f"{filler}{r}") for r in range(1, n + 1)]
A = ranked({1: "x1", 40: "y2", 100: "y1"}, 100, "a")
B = ranked({2: "x2", 38: "y1", 98: "y2"}, 100, "b")
probe = [d for d in rrf([A, B]) if d in {"x1", "y1", "x2", "y2"}]
assert probe == ["y1", "x1", "y2", "x2"], f"{probe}: use 1 / (k + rank + 1) with k=60"

def hybrid_ids(query, k=5):
    return rrf([dense_ids(query, 10), bm25_ids(query, 10)])[:k]

for q in ["what does ERR_4290 mean", "am I allowed to work from home every day"]:
    print(q, "->", [i.split('#')[0] for i in hybrid_ids(q, 3)])

Hybrid gets the exact-code query *and* the paraphrase. In notebook 05 we'll
confirm with numbers that it beats either method alone across the whole query
set.
